# S14b — MI-LiteXGB en CICIoT2023: balanceo y falsos positivos
Experimento complementario al cuaderno 07. En el 07, el balanceo *T* de MI-LiteXGB se eligió por macro-F1 en CV y quedó en
T = 20 000 en 8 de 10 REP, con una tasa de falsos positivos (TFP) de ≈ 12 %: el sobremuestreo infla las clases pequeñas y les
asigna paquetes benignos. Aquí se mide ese efecto y se prueban dos correcciones **fijadas antes de ver resultados**:

| Variante | Qué cambia | Reentrena |
|---|---|---|
| **Curva T** | MI-LiteXGB con cada *T* fijo (sin balanceo, 2 000, 20 000) evaluado en prueba; es descriptiva, no elige nada | Sí |
| **CV con restricción de TFP** | Igual que el 07, pero solo se admiten los *T* cuya TFP en CV no supere a la de «sin balanceo»; entre esos, el de mayor macro-F1 en CV | Sí |
| **Corrección de priores** | El mismo modelo que eligió el 07 (mismo *T*); solo se reescalan sus probabilidades a la distribución de clases original del entrenamiento (Saerens et al., 2002) | No |

Mismas divisiones (semilla 1000 + r), mismo *k* = 25 y mismos atributos por REP que el 07. Necesita que el 07 haya terminado
(lee `cic_muestra.parquet`, `A2_eleccion_k.json` y los `A2_arboles/REP-XX/resultado.json` de Drive). Solo CPU; ≈ 3–5 min por REP.
Si la sesión se corta, vuelve a ejecutar todo: retoma por REP.

Al final, **comparte `S14_XGB_TFP_resultados.json`** (queda en `mejoras/S14_CICIoT2023/`).

In [ ]:
import os
# --- Qué se ejecuta ---------------------------------------------------------------------------------
EJECUTAR = {'A1': False, 'A2': False, 'A7': True}   # solo A7; A0 reutiliza la muestra ya guardada
REPS = list(range(1, 11))
REANUDAR = True              # reutiliza lo ya guardado en Drive (por parte y por REP)
MODO_PRUEBA = os.environ.get('S14_PRUEBA') == '1'   # datos sintéticos pequeños, sin Drive ni Kaggle (solo para probar el cuaderno)

# --- Rutas ------------------------------------------------------------------------------------------
CARPETA = os.environ.get('CARPETA_ARTICULO', '/content/drive/MyDrive/Articulo_Grupal_EdgeIIoT')
DIR_CSV_DRIVE = os.path.join(CARPETA, 'CICIoT2023_csv')     # opcional: CSV originales copiados a mano
KAGGLE_SLUG = 'madhavmalhotra/unb-cic-iot-dataset'           # espejo en Kaggle de los CSV originales (part-*.csv)
DIR_X = os.path.join(CARPETA, 'mejoras', 'S14_CICIoT2023')    # salida
DIR_TMP = os.environ.get('DIR_TMP_S14', '/content/tmp_s14')

# --- Muestra y tarea (fijadas antes de mirar resultados) --------------------------------------------
N_ARCHIVOS = 5               # CSV usados de los ~169 (a intervalos regulares de la lista ordenada)
TAREA = '8clases'            # '8clases' (Benign + 7 categorías de la UNB) o '34clases' (cada ataque)
SEMILLA_BASE = 1000          # REP r usa la semilla 1000 + r, como en el artículo
TEST_SIZE = 0.30
# --- Igual que el artículo ---------------------------------------------------------------------------
MI_SUBMUESTRA = 200_000      # paquetes de entrenamiento para estimar la IM
MI_MAX_DISCRETO = 64         # atributos con ≤ 64 valores se usan tal cual; el resto, 32 bins por cuantiles
MI_BINS = 32
K_GRID = [5, 10, 15, 20, 25, 30]
T_GRID = [2000, 20000, None] # None = sin balanceo (distribución natural)
TOLERANCIA = 0.99            # regla de parsimonia: CV ≥ 99 % del mejor y, entre esos, el modelo más pequeño
CV_SUBMUESTRA = 400_000      # filas de entrenamiento (estratificadas) para la CV de k y de T (por costo)
SMOTE_K = 5
RF_CFG = dict(arboles=25, profundidad=15)                    # MI-LiteRF del artículo
XGB_CFG = dict(arboles=50, profundidad=4, lr=0.1)           # MI-LiteXGB del artículo
TCN_CFG = dict(lr=1e-3, epocas=50, lote=128, dilataciones=[1, 2, 4, 8], kernel=1, filtros=26, dropout=0.1, val_split=0.1)
# --- Auditoría (A1) ------------------------------------------------------------------------------------
UMBRAL_PUREZA = 0.999        # marcador candidato si una sola columna acierta Normal/Ataque en ≥ 99,9 % de las filas
UMBRAL_IM_REL = 0.90         # ... o si su IM con Normal/Ataque alcanza el 90 % de H(Y_b)
# --- Latencia (A5) --------------------------------------------------------------------------------------
N_LOTE1 = 10_000
CALENTAMIENTO = 500
ALFA = 0.05

if MODO_PRUEBA:   # tamaños mínimos para probar el flujo completo en minutos
    CARPETA = os.environ.get('CARPETA_PRUEBA', '/tmp/prueba_s14')
    DIR_X = os.path.join(CARPETA, 'mejoras', 'S14_CICIoT2023'); DIR_CSV_DRIVE = os.path.join(CARPETA, 'CICIoT2023_csv')
    DIR_TMP = os.path.join(CARPETA, 'tmp'); REPS = [1, 2]; MI_SUBMUESTRA = 20_000; CV_SUBMUESTRA = 20_000
    K_GRID = [5, 10, 15]; TCN_CFG = dict(TCN_CFG, epocas=2, lote=256); N_LOTE1 = 300; CALENTAMIENTO = 20
os.makedirs(DIR_TMP, exist_ok=True)
print('REPS:', REPS, '| partes:', [p for p, v in EJECUTAR.items() if v], '| tarea:', TAREA, '| salida:', DIR_X)

## 0. Drive, versiones y librerías
Se fijan scikit-learn, imbalanced-learn y XGBoost en las versiones del artículo. Si Colab trae otras, la celda las instala
y pide reiniciar la sesión una vez.

In [ ]:
import sys, subprocess, json, time, platform, datetime, io, gc, glob, gzip, shutil, warnings, math
import importlib.metadata as im
from types import SimpleNamespace
EN_COLAB = 'google.colab' in sys.modules
if EN_COLAB and not MODO_PRUEBA and not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
if not MODO_PRUEBA and not os.path.isdir(CARPETA):
    raise SystemExit(f'⛔ No encuentro {CARPETA}. Revisa que el Drive montado sea el de la cuenta dueña de Articulo_Grupal_EdgeIIoT.')
os.makedirs(DIR_X, exist_ok=True)

def version(paquete):
    try:
        return im.version(paquete)
    except im.PackageNotFoundError:
        return None

V_ARTICULO = {'scikit-learn': '1.6.1', 'imbalanced-learn': '0.14.2', 'xgboost': '3.4.1'}
if EN_COLAB and not MODO_PRUEBA:
    distintas = {p: version(p) for p, v in V_ARTICULO.items() if version(p) != v}
    if distintas:
        ya = [m for m in ('sklearn', 'imblearn', 'xgboost') if m in sys.modules]
        print('Instalando las versiones del artículo (ahora hay', distintas, ')')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *[f'{p}=={v}' for p, v in V_ARTICULO.items()]], check=True)
        if ya:
            raise SystemExit('⛔ Se cambiaron librerías ya cargadas: Entorno de ejecución → Reiniciar sesión, y vuelve a ejecutar todo.')
    faltan = [p for p in ('onnx', 'onnxruntime', 'skl2onnx', 'onnxmltools', 'kagglehub', 'pyarrow') if version(p) is None]
    if faltan:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *faltan], check=True)
    if version('tf2onnx') is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'tf2onnx'], check=True)

import numpy as np, pandas as pd, joblib
import sklearn, imblearn, xgboost, scipy
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from scipy import stats
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

def cpu_modelo():
    try:
        for l in open('/proc/cpuinfo'):
            if l.startswith('model name'):
                return l.split(':', 1)[1].strip()
    except Exception:
        pass
    return platform.processor()

def hay_gpu():
    return bool(shutil.which('nvidia-smi')) and 'GPU' in subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout

ENTORNO = {'fecha_hora': datetime.datetime.now().strftime('%d/%m/%Y %H:%M'), 'procesador': cpu_modelo(),
           'nucleos_logicos': os.cpu_count(), 'gpu': hay_gpu(), 'python': platform.python_version(),
           'numpy': np.__version__, 'pandas': pd.__version__, 'scikit_learn': sklearn.__version__,
           'imbalanced_learn': imblearn.__version__, 'xgboost': xgboost.__version__, 'scipy': scipy.__version__,
           'tensorflow': version('tensorflow'), 'onnxruntime': version('onnxruntime'), 'modo_prueba': MODO_PRUEBA}
print(json.dumps(ENTORNO, indent=1))

## 1. Utilidades (las mismas del artículo)

In [ ]:
def guardar_json(obj, ruta):
    os.makedirs(os.path.dirname(ruta), exist_ok=True)
    with open(ruta, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=1,
                  default=lambda o: o.item() if hasattr(o, 'item') else (o.tolist() if hasattr(o, 'tolist') else str(o)))

def leer_json(ruta):
    with open(ruta, encoding='utf-8') as f:
        return json.load(f)

def hay(ruta):
    return REANUDAR and os.path.exists(ruta)

def correr(parte):
    if not EJECUTAR.get(parte):
        print(f'⏭️ {parte} desactivada en EJECUTAR.')
        return False
    return True

def entropia(*codigos):
    c = codigos[0].astype(np.int64)
    for otro in codigos[1:]:
        c = c * (int(otro.max()) + 1) + otro.astype(np.int64)
    _, cnt = np.unique(c, return_counts=True)
    p = cnt / cnt.sum()
    return float(-(p * np.log2(p)).sum())

def discretizar(x):
    u = np.unique(x)
    if len(u) <= MI_MAX_DISCRETO:
        return np.searchsorted(u, x)
    bordes = np.unique(np.quantile(x, np.linspace(0, 1, MI_BINS + 1)[1:-1]))
    return np.digitize(x, bordes)

def info_mutua(xd, y):
    return entropia(y) + entropia(xd) - entropia(xd, y)

def pureza(xd, yb):
    """Exactitud de predecir Normal/Ataque con la etiqueta mayoritaria de cada valor (prueba de una columna de Galala)."""
    tab = pd.crosstab(xd, yb)
    return float(tab.max(axis=1).sum() / tab.values.sum())

def balancear(Xs, ys, T, semilla):
    cnt = np.bincount(ys, minlength=N_CLASES)
    bajo = {c: T for c in range(N_CLASES) if cnt[c] > T}
    if bajo:
        Xs, ys = RandomUnderSampler(sampling_strategy=bajo, random_state=semilla).fit_resample(Xs, ys)
    cnt = np.bincount(ys, minlength=N_CLASES)
    sobre = {c: T for c in range(N_CLASES) if 0 < cnt[c] < T}
    if sobre:
        kn = max(1, min(SMOTE_K, int(cnt[list(sobre)].min()) - 1))
        Xs, ys = SMOTE(sampling_strategy=sobre, k_neighbors=kn, random_state=semilla).fit_resample(Xs, ys)
    Xs, ys = TomekLinks(sampling_strategy='all', n_jobs=-1).fit_resample(Xs, ys)
    return np.ascontiguousarray(Xs, dtype=np.float32), ys

def datos_T(Xs, ys, T, semilla):
    if not T:
        return np.ascontiguousarray(Xs, dtype=np.float32), ys
    return balancear(Xs, ys, int(T), semilla)

def T_txt(T):
    return 'sin balanceo' if not T else f'{int(T):,}'.replace(',', ' ')

def indicadores(y_real, y_pred):
    cm = confusion_matrix(y_real, y_pred, labels=range(N_CLASES))
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    den = 2 * tp + fp + fn
    f1 = np.divide(2 * tp, den, out=np.zeros_like(tp), where=den > 0) * 100
    TN = cm[NORMAL, NORMAL]; FP = cm[NORMAL].sum() - TN
    ataques = [c for c in range(N_CLASES) if c != NORMAL]
    miss = cm[ataques, NORMAL].sum() / max(cm[ataques].sum(), 1) * 100
    en = []
    for c in ataques:
        if cm[c].sum() > 0:
            m = cm[c, NORMAL] / cm[c].sum()
            if m < 1:
                en.append(m / (1 - m))
    return {'f1_macro': float(f1.mean()), 'f1_por_clase': dict(zip(CLASES, f1.tolist())),
            'tfp': float(FP / max(FP + TN, 1) * 100), 'falsas_alarmas': int(FP), 'ataques_como_normal_pct': float(miss),
            'E_N_registros': float(np.mean(en)) if en else None,
            'exactitud': float(accuracy_score(y_real, y_pred) * 100),
            'f1_ponderado': float(f1_score(y_real, y_pred, average='weighted') * 100),
            'matriz_confusion': cm.tolist()}

def f1_macro(y_real, y_pred):
    return f1_score(y_real, y_pred, average='macro', labels=range(N_CLASES), zero_division=0) * 100

def holm(pvals):
    p = np.asarray(pvals, float); m = len(p)
    orden = np.argsort(p, kind='mergesort'); ajust = np.empty(m); acumulado = 0.0
    for i, idx in enumerate(orden):
        acumulado = max(acumulado, min(1.0, (m - i) * p[idx])); ajust[idx] = acumulado
    return ajust

def t_corregida(d, n_tr, n_te):
    """t remuestreada corregida (Nadeau y Bengio, 2003)."""
    d = np.asarray(d, float); J = len(d); m = float(d.mean())
    s2 = float(d.var(ddof=1)) if J > 1 else 0.0
    se = np.sqrt((1 / J + n_te / n_tr) * s2)
    t = m / se if se > 0 else (0.0 if m == 0 else np.inf)
    h = stats.t.ppf(0.975, max(J - 1, 1)) * se
    return {'media_d': m, 't_corr': float(t), 'gl': J - 1, 'p_corr': float(2 * stats.t.sf(abs(t), max(J - 1, 1))),
            'ic95_corr': [round(float(m - h), 4), round(float(m + h), 4)]}

def mostrar(df):
    try:
        display(df)
    except NameError:
        print(df.to_string())

def ms(x):
    x = np.asarray(x, float)
    return [float(x.mean()), float(x.std(ddof=1)) if len(x) > 1 else 0.0]

def kb_joblib(modelo):
    b = io.BytesIO(); joblib.dump(modelo, b); return b.tell() / 1024

def nuevo_rf(semilla):
    return RandomForestClassifier(RF_CFG['arboles'], max_depth=RF_CFG['profundidad'], n_jobs=-1, random_state=semilla)

def nuevo_xgb(semilla):
    return XGBClassifier(n_estimators=XGB_CFG['arboles'], max_depth=XGB_CFG['profundidad'], learning_rate=XGB_CFG['lr'],
                         tree_method='hist', device='cpu', n_jobs=-1, random_state=semilla, eval_metric='mlogloss')
FABRICA = {'rf': nuevo_rf, 'xgb': nuevo_xgb}

## A0. Datos: descarga, muestra de archivos y limpieza
Se usan `N_ARCHIVOS` CSV originales elegidos a intervalos regulares de la lista ordenada (cada archivo mezcla todas las clases).
Limpieza igual que en Edge-IIoTset: se quitan filas con valores faltantes o no finitos y luego los duplicados.
Si el CSV trae una marca de tiempo (`ts`), se retira, como `frame.time` en Edge-IIoTset.

In [ ]:
CAT_34_A_8 = {
    'benigntraffic': 'Benign',
    **{k: 'DDoS' for k in ['ddos-rstfinflood', 'ddos-pshack_flood', 'ddos-syn_flood', 'ddos-udp_flood', 'ddos-tcp_flood',
                            'ddos-icmp_flood', 'ddos-synonymousip_flood', 'ddos-ack_fragmentation', 'ddos-udp_fragmentation',
                            'ddos-icmp_fragmentation', 'ddos-slowloris', 'ddos-http_flood']},
    **{k: 'DoS' for k in ['dos-udp_flood', 'dos-syn_flood', 'dos-tcp_flood', 'dos-http_flood']},
    **{k: 'Mirai' for k in ['mirai-greeth_flood', 'mirai-greip_flood', 'mirai-udpplain']},
    **{k: 'Recon' for k in ['recon-pingsweep', 'recon-osscan', 'recon-portscan', 'vulnerabilityscan', 'recon-hostdiscovery']},
    **{k: 'Spoofing' for k in ['dns_spoofing', 'mitm-arpspoofing']},
    **{k: 'Web' for k in ['browserhijacking', 'backdoor_malware', 'xss', 'uploading_attack', 'sqlinjection', 'commandinjection']},
    'dictionarybruteforce': 'BruteForce'}

def categoria(etiqueta):
    e = str(etiqueta).strip().lower()
    if e in CAT_34_A_8:
        return CAT_34_A_8[e]
    for pref, cat in (('ddos', 'DDoS'), ('dos', 'DoS'), ('mirai', 'Mirai'), ('recon', 'Recon'), ('benign', 'Benign')):
        if e.startswith(pref):
            return cat
    return None

def datos_sinteticos(ruta, n=60_000, semilla=7):
    """Solo para MODO_PRUEBA: CSV con el formato de CICIoT2023 (46 atributos + label) y 34 etiquetas."""
    rng = np.random.default_rng(semilla)
    cols = ['flow_duration', 'Header_Length', 'Protocol Type', 'Duration', 'Rate', 'Srate', 'Drate', 'fin_flag_number',
            'syn_flag_number', 'rst_flag_number', 'psh_flag_number', 'ack_flag_number', 'ece_flag_number', 'cwr_flag_number',
            'ack_count', 'syn_count', 'fin_count', 'urg_count', 'rst_count', 'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP', 'SSH', 'IRC',
            'TCP', 'UDP', 'DHCP', 'ARP', 'ICMP', 'IPv', 'LLC', 'Tot sum', 'Min', 'Max', 'AVG', 'Std', 'Tot size', 'IAT', 'Number',
            'Magnitue', 'Radius', 'Covariance', 'Variance', 'Weight']
    etiquetas = list(CAT_34_A_8.keys())
    p = np.array([0.03] + [0.9 / 33] * 33); p = p / p.sum()
    nombres = {k: k for k in etiquetas}
    nombres.update({'benigntraffic': 'BenignTraffic', 'ddos-rstfinflood': 'DDoS-RSTFINFlood', 'xss': 'XSS'})
    y = rng.choice(len(etiquetas), size=n, p=p)
    X = rng.normal(size=(n, len(cols))) + (y[:, None] % 7) * 0.4 * (np.arange(len(cols)) % 5 == 0)
    X[:, cols.index('IAT')] = 8.3e7 + y * 1e4 + rng.normal(scale=50, size=n)   # marcador de captura, como sospechamos
    X[:, cols.index('Protocol Type')] = rng.integers(0, 17, size=n)
    df = pd.DataFrame(X.astype(np.float32), columns=cols)
    df['label'] = [nombres[etiquetas[i]] for i in y]
    df.iloc[:50, 4] = np.inf                                                           # filas no finitas
    df = pd.concat([df, df.iloc[:200]], ignore_index=True)                            # duplicados
    os.makedirs(ruta, exist_ok=True)
    df = df.sample(frac=1, random_state=1).reset_index(drop=True)
    for i, idx in enumerate(np.array_split(np.arange(len(df)), 8)):
        df.iloc[idx].to_csv(os.path.join(ruta, f'part-{i:05d}-prueba.csv'), index=False)

def localizar_csv():
    archivos = sorted(glob.glob(os.path.join(DIR_CSV_DRIVE, '**', '*.csv'), recursive=True))
    if archivos:
        return archivos, f'Drive ({DIR_CSV_DRIVE})'
    if MODO_PRUEBA:
        datos_sinteticos(DIR_CSV_DRIVE)
        return sorted(glob.glob(os.path.join(DIR_CSV_DRIVE, '*.csv'))), 'sintéticos (MODO_PRUEBA)'
    try:
        import kagglehub
        print(f'Descargando {KAGGLE_SLUG} con kagglehub (≈ 14 GB; solo la primera vez)…')
        ruta = kagglehub.dataset_download(KAGGLE_SLUG)
    except Exception as e:
        raise SystemExit(f'⛔ No se pudo descargar de Kaggle ({type(e).__name__}: {str(e)[:200]}).\n'
                         '   Opción 1: en Colab, ícono de llave (Secretos) → agrega KAGGLE_USERNAME y KAGGLE_KEY y vuelve a ejecutar.\n'
                         f'   Opción 2: copia a mano algunos part-*.csv de CICIoT2023 a {DIR_CSV_DRIVE} y vuelve a ejecutar.')
    return sorted(glob.glob(os.path.join(ruta, '**', '*.csv'), recursive=True)), f'Kaggle ({KAGGLE_SLUG})'

def columna_etiqueta(cols):
    for c in cols:
        if c.strip().lower() == 'label':
            return c
    return None

RUTA_MUESTRA = os.path.join(DIR_X, 'cic_muestra.parquet')
RUTA_A0 = os.path.join(DIR_X, 'A0_datos.json')
if hay(RUTA_MUESTRA) and hay(RUTA_A0):
    A0 = leer_json(RUTA_A0)
    print('✔ Muestra ya preparada:', A0['archivos_usados'], 'archivos ·', f"{A0['filas_depuradas']:,} filas")
else:
    t0 = time.time()
    todos, origen = localizar_csv()
    validos = []
    for a in todos:
        try:
            c = pd.read_csv(a, nrows=0).columns
        except Exception:
            continue
        if columna_etiqueta(c) is not None and len(c) >= 40:
            validos.append(a)
    if not validos:
        raise SystemExit(f'⛔ No encontré CSV de CICIoT2023 (con columna «label» y ≥ 40 columnas) en {origen}.')
    elegidos = sorted({validos[i] for i in np.linspace(0, len(validos) - 1, min(N_ARCHIVOS, len(validos))).round().astype(int)})
    print(f'{origen}: {len(validos)} CSV válidos; se usan {len(elegidos)}:', [os.path.basename(a) for a in elegidos])
    partes = []
    for a in elegidos:
        d = pd.read_csv(a, low_memory=False)
        d.columns = [c.strip() for c in d.columns]
        partes.append(d)
    df = pd.concat(partes, ignore_index=True); del partes
    col_y = columna_etiqueta(df.columns)
    n_bruto = len(df)
    retiradas = [c for c in df.columns if c.lower() in ('ts', 'timestamp')]
    df = df.drop(columns=retiradas)
    atributos = [c for c in df.columns if c != col_y]
    for c in atributos:
        if not pd.api.types.is_numeric_dtype(df[c]):
            df[c] = pd.to_numeric(df[c], errors='coerce')
    df[atributos] = df[atributos].astype(np.float32)
    df = df.replace([np.inf, -np.inf], np.nan).dropna(axis=0, how='any')
    n_finitas = len(df)
    df = df.drop_duplicates(keep='first')
    n_dep = len(df)
    df['etiqueta34'] = df[col_y].astype(str).str.strip()
    df['categoria'] = df['etiqueta34'].map(categoria)
    sin_cat = sorted(df.loc[df['categoria'].isna(), 'etiqueta34'].unique().tolist())
    if sin_cat:
        print('⚠️ Etiquetas sin categoría (se excluyen):', sin_cat)
        df = df[df['categoria'].notna()]
    df = df.drop(columns=[col_y]).reset_index(drop=True)
    df.to_parquet(RUTA_MUESTRA, index=False)
    A0 = {'origen': origen, 'archivos_validos': len(validos), 'archivos_usados': len(elegidos),
          'archivos': [os.path.basename(a) for a in elegidos], 'filas_brutas': int(n_bruto), 'filas_finitas': int(n_finitas),
          'filas_depuradas': int(len(df)), 'duplicados_eliminados': int(n_finitas - n_dep), 'columnas_retiradas': retiradas,
          'atributos': atributos, 'etiquetas_sin_categoria': sin_cat,
          'clases_34': df['etiqueta34'].value_counts().to_dict(), 'clases_8': df['categoria'].value_counts().to_dict(),
          'minutos': round((time.time() - t0) / 60, 1)}
    guardar_json(A0, RUTA_A0)
    del df; gc.collect()
    print(f"✔ {A0['filas_depuradas']:,} filas depuradas de {A0['filas_brutas']:,} ({A0['duplicados_eliminados']:,} duplicados) "
          f"en {A0['minutos']} min")

D = pd.read_parquet(RUTA_MUESTRA)
ATRIBUTOS = [c for c in D.columns if c not in ('etiqueta34', 'categoria')]
col_clase = 'categoria' if TAREA == '8clases' else 'etiqueta34'
CLASES = sorted(D[col_clase].unique().tolist())
CLASES = ['Benign'] + [c for c in CLASES if c != 'Benign'] if 'Benign' in CLASES else \
         (['BenignTraffic'] + [c for c in CLASES if c != 'BenignTraffic'])
N_CLASES = len(CLASES)
NORMAL = 0
X_TODO = D[ATRIBUTOS].to_numpy(np.float32)
Y_TODO = D[col_clase].map({c: i for i, c in enumerate(CLASES)}).to_numpy(np.int64)
YB_TODO = (Y_TODO != NORMAL).astype(np.int64)
del D; gc.collect()
print(f'{len(Y_TODO):,} filas · {len(ATRIBUTOS)} atributos · {N_CLASES} clases')
print(pd.Series(np.bincount(Y_TODO, minlength=N_CLASES), index=CLASES).to_string())

## A1. Auditoría de entropía (sin entrenar)
Para cada atributo: IM con Normal/Ataque y con la clase, comparada con la entropía de la etiqueta, y pureza de una columna.
Un atributo es **marcador candidato** si su pureza es ≥ 0,999 o su IM alcanza el 90 % de H(Y_b) (reglas fijadas antes).
Conflictos de etiqueta: filas con el mismo vector de atributos y etiquetas distintas; el oráculo de mayoría da una cota (en la muestra).

In [ ]:
RUTA_A1 = os.path.join(DIR_X, 'A1_auditoria.json')
if correr('A1'):
    if hay(RUTA_A1):
        A1 = leer_json(RUTA_A1)
    else:
        t0 = time.time()
        H_yb, H_y = entropia(YB_TODO), entropia(Y_TODO)
        filas = []
        for j, a in enumerate(ATRIBUTOS):
            xd = discretizar(X_TODO[:, j])
            filas.append({'atributo': a, 'valores_distintos': int(len(np.unique(X_TODO[:, j]))),
                          'im_binaria': info_mutua(xd, YB_TODO), 'im_clase': info_mutua(xd, Y_TODO),
                          'pureza': pureza(xd, YB_TODO)})
        T1 = pd.DataFrame(filas)
        T1['im_binaria_rel'] = T1['im_binaria'] / H_yb
        T1['im_clase_rel'] = T1['im_clase'] / H_y
        T1['marcador'] = (T1['pureza'] >= UMBRAL_PUREZA) | (T1['im_binaria_rel'] >= UMBRAL_IM_REL)
        T1 = T1.sort_values('im_binaria', ascending=False).reset_index(drop=True)
        # Conflictos de etiqueta con todos los atributos
        h = pd.util.hash_pandas_object(pd.DataFrame(X_TODO), index=False).to_numpy()
        tab = pd.DataFrame({'h': h, 'y': Y_TODO})
        n_etq = tab.groupby('h')['y'].nunique()
        conflicto = tab['h'].map(n_etq).to_numpy() > 1
        cnt = tab.groupby(['h', 'y']).size().reset_index(name='n').sort_values(['h', 'n'], ascending=[True, False])
        cnt = cnt.drop_duplicates('h')
        mayoria = pd.Series(cnt['y'].to_numpy(), index=cnt['h'].to_numpy())
        pred_or = tab['h'].map(mayoria).to_numpy()
        ind_or = indicadores(Y_TODO, pred_or)
        A1 = {'H_binaria': H_yb, 'H_clase': H_y, 'pct_normal': float((Y_TODO == NORMAL).mean() * 100),
              'pureza_base': float(max((YB_TODO == 0).mean(), (YB_TODO == 1).mean())),
              'atributos': T1.to_dict(orient='records'), 'marcadores': T1.loc[T1['marcador'], 'atributo'].tolist(),
              'conflictos_pct': float(conflicto.mean() * 100), 'vectores_distintos': int(len(n_etq)),
              'oraculo': {k: ind_or[k] for k in ('f1_macro', 'tfp', 'ataques_como_normal_pct', 'exactitud')},
              'umbrales': {'pureza': UMBRAL_PUREZA, 'im_rel': UMBRAL_IM_REL}, 'segundos': round(time.time() - t0, 1)}
        guardar_json(A1, RUTA_A1)
    T1 = pd.DataFrame(A1['atributos'])
    print(f"H(Y_b) = {A1['H_binaria']:.4f} bits (Normal {A1['pct_normal']:.2f} %) · H(Y) = {A1['H_clase']:.4f} bits · "
          f"pureza de referencia (siempre la mayoritaria) = {A1['pureza_base']:.4f}")
    print('Marcadores candidatos:', A1['marcadores'] or 'ninguno')
    print(f"Conflictos de etiqueta: {A1['conflictos_pct']:.2f} % de las filas · oráculo: F1 macro {A1['oraculo']['f1_macro']:.2f} % · "
          f"TFP {A1['oraculo']['tfp']:.3f} %")
    mostrar(T1.head(15).round(4))

## A2. MI-LiteRF y MI-LiteXGB en 10 repeticiones
Por REP: división estratificada 70/30 (semilla 1000 + r), escalado min–max con el entrenamiento, ranking de IM en 200 000 filas de
entrenamiento. *k* se elige **una vez**, por CV en la REP-01 (XGBoost sin balanceo, regla de parsimonia) y se mantiene.
El balanceo *T* se elige por CV en cada REP y para cada modelo. La CV usa una submuestra estratificada de 400 000 filas de entrenamiento.

In [ ]:
def dividir(r):
    semilla = SEMILLA_BASE + r
    idx_tr, idx_te = train_test_split(np.arange(len(Y_TODO)), test_size=TEST_SIZE, stratify=Y_TODO, random_state=semilla)
    esc = MinMaxScaler().fit(X_TODO[idx_tr])
    return SimpleNamespace(r=r, codigo=f'REP-{r:02d}', semilla=semilla, idx_tr=idx_tr, idx_te=idx_te, esc=esc,
                           Xtr=esc.transform(X_TODO[idx_tr]).astype(np.float32), Xte=esc.transform(X_TODO[idx_te]).astype(np.float32),
                           ytr=Y_TODO[idx_tr], yte=Y_TODO[idx_te])

def submuestra(y, n, semilla):
    if len(y) <= n:
        return np.arange(len(y))
    idx, _ = train_test_split(np.arange(len(y)), train_size=n, stratify=y, random_state=semilla)
    return np.sort(idx)

def ranking_im(Dv, excluir=()):
    sub = submuestra(Dv.ytr, MI_SUBMUESTRA, Dv.semilla)
    Xs, ys = Dv.Xtr[sub], Dv.ytr[sub]
    im_ = []
    for j, a in enumerate(ATRIBUTOS):
        im_.append(-1.0 if a in excluir else info_mutua(discretizar(Xs[:, j]), ys))
    orden = np.argsort(-np.asarray(im_), kind='mergesort')
    orden = [j for j in orden if ATRIBUTOS[j] not in excluir]
    return orden, im_

def cv_puntaje(Dv, cols, fabrica, T, sub):
    skf = StratifiedKFold(3, shuffle=True, random_state=Dv.semilla)
    X, y = Dv.Xtr[sub][:, cols], Dv.ytr[sub]
    f1s, tam = [], []
    for k_, (a, b) in enumerate(skf.split(X, y)):
        Xb, yb = datos_T(X[a], y[a], T, Dv.semilla + k_)
        m = fabrica(Dv.semilla).fit(Xb, yb)
        f1s.append(f1_macro(y[b], m.predict(X[b]))); tam.append(kb_joblib(m))
    return float(np.mean(f1s)), float(np.mean(tam))

def regla_parsimonia(tabla, clave):
    mejor = tabla['f1_cv'].max()
    cand = tabla[tabla['f1_cv'] >= TOLERANCIA * mejor]
    return cand.sort_values(['tamano_kb', clave]).iloc[0]

RUTA_K = os.path.join(DIR_X, 'A2_eleccion_k.json')
def elegir_k():
    if hay(RUTA_K):
        return leer_json(RUTA_K)
    Dv = dividir(1); orden, _ = ranking_im(Dv); sub = submuestra(Dv.ytr, CV_SUBMUESTRA, Dv.semilla)
    filas = []
    for k in K_GRID:
        f1, tam = cv_puntaje(Dv, orden[:k], nuevo_xgb, None, sub)
        filas.append({'k': k, 'f1_cv': f1, 'tamano_kb': tam}); print(f'   k = {k}: F1 CV {f1:.2f} · {tam:.0f} KB')
    tabla = pd.DataFrame(filas)
    # parsimonia sobre k: el menor k con CV ≥ 99 % del mejor (como en el artículo)
    mejor = tabla['f1_cv'].max()
    k = int(tabla[tabla['f1_cv'] >= TOLERANCIA * mejor]['k'].min())
    res = {'k': k, 'tabla': filas, 'atributos_rep01': [ATRIBUTOS[j] for j in orden[:k]]}
    guardar_json(res, RUTA_K)
    return res

def a2_rep(r):
    dir_r = os.path.join(DIR_X, 'A2_arboles', f'REP-{r:02d}'); ruta = os.path.join(dir_r, 'resultado.json')
    if hay(ruta):
        return leer_json(ruta)
    os.makedirs(dir_r, exist_ok=True); t_ini = time.time()
    K = elegir_k()['k']
    Dv = dividir(r); orden, im_ = ranking_im(Dv); cols = orden[:K]
    sub = submuestra(Dv.ytr, CV_SUBMUESTRA, Dv.semilla)
    res = {'rep': r, 'semilla': Dv.semilla, 'k': K, 'atributos_k': [ATRIBUTOS[j] for j in cols],
           'ranking': [{'atributo': ATRIBUTOS[j], 'im': float(im_[j])} for j in orden], 'n_train': int(len(Dv.ytr)),
           'n_test': int(len(Dv.yte)), 'modelos': {}}
    for nombre in ('rf', 'xgb'):
        filas = []
        for T in T_GRID:
            f1, tam = cv_puntaje(Dv, cols, FABRICA[nombre], T, sub)
            filas.append({'T': T if T else 0, 'f1_cv': f1, 'tamano_kb': tam})
        tabla = pd.DataFrame(filas)
        T = int(regla_parsimonia(tabla, 'T')['T']) or None
        Xb, yb = datos_T(Dv.Xtr[:, cols], Dv.ytr, T, Dv.semilla)
        t0 = time.perf_counter(); m = FABRICA[nombre](Dv.semilla).fit(Xb, yb); t_ent = time.perf_counter() - t0
        pred = m.predict(Dv.Xte[:, cols])
        res['modelos'][nombre] = {'T': T or 0, 'cv': filas, 'tiempo_entrenamiento_s': t_ent, 'tamano_kb': kb_joblib(m),
                                  **indicadores(Dv.yte, pred)}
        if r == 1:
            joblib.dump(m, os.path.join(dir_r, f'{nombre}.joblib'))
        print(f"   {nombre.upper()}: T = {T_txt(T)} · F1 macro {res['modelos'][nombre]['f1_macro']:.2f} % · "
              f"TFP {res['modelos'][nombre]['tfp']:.3f} % · {t_ent:.0f} s")
    if r == 1:
        np.savez(os.path.join(dir_r, 'escalador.npz'), min_=Dv.esc.min_, scale_=Dv.esc.scale_, cols=np.asarray(cols))
    res['segundos'] = round(time.time() - t_ini, 1)
    guardar_json(res, ruta)
    return res

if correr('A2'):
    print('Elección de k (REP-01, CV):'); EK = elegir_k(); print(f"   k = {EK['k']}")
    A2 = {}
    for r in REPS:
        print(f'REP-{r:02d}'); A2[r] = a2_rep(r)

## A7. Balanceo y falsos positivos en MI-LiteXGB
Reglas fijadas antes de ver resultados:
* **CV con restricción de TFP**: CV de 3 particiones en la misma submuestra de 400 000 filas del 07. Un *T* es admisible si su TFP media
  en CV es ≤ la de «sin balanceo»; se elige el admisible de mayor macro-F1 en CV («sin balanceo» siempre es admisible).
* **Corrección de priores**: p'(c|x) ∝ p(c|x) · π(c) / π_b(c), con π las proporciones de clase del entrenamiento original y π_b las del
  entrenamiento balanceado; se predice la clase de mayor p'. Si el 07 eligió «sin balanceo», la corrección no cambia nada.
* Pruebas pareadas (t remuestreada corregida, Holm por indicador) de cada variante contra MI-LiteXGB del 07 y contra MI-LiteRF del 07.

In [ ]:
DIR_A7 = os.path.join(DIR_X, 'A7_xgb_tfp')

def cv_f1_tfp(Dv, cols, T, sub):
    skf = StratifiedKFold(3, shuffle=True, random_state=Dv.semilla)
    X, y = Dv.Xtr[sub][:, cols], Dv.ytr[sub]
    f1s, tfps = [], []
    for k_, (a, b) in enumerate(skf.split(X, y)):
        Xb, yb = datos_T(X[a], y[a], T, Dv.semilla + k_)
        ind = indicadores(y[b], nuevo_xgb(Dv.semilla).fit(Xb, yb).predict(X[b]))
        f1s.append(ind['f1_macro']); tfps.append(ind['tfp'])
    return float(np.mean(f1s)), float(np.mean(tfps))

def corregir_priores(P, pi, pi_b):
    w = np.divide(pi, pi_b, out=np.zeros_like(pi), where=pi_b > 0)
    Q = P * w
    return Q.argmax(1)

def a7_rep(r):
    ruta = os.path.join(DIR_A7, f'REP-{r:02d}.json')
    if hay(ruta):
        return leer_json(ruta)
    t_ini = time.time()
    ref = leer_json(os.path.join(DIR_X, 'A2_arboles', f'REP-{r:02d}', 'resultado.json'))
    K = elegir_k()['k']; Dv = dividir(r)
    cols = [ATRIBUTOS.index(a) for a in ref['atributos_k']]          # exactamente los atributos del 07
    assert len(cols) == K
    T07 = ref['modelos']['xgb']['T'] or None
    pi = np.bincount(Dv.ytr, minlength=N_CLASES) / len(Dv.ytr)
    res = {'rep': r, 'semilla': Dv.semilla, 'k': K, 'T_07': T07 or 0, 'curva_T': {}, 'priores': {}}
    # 1) Curva T: un modelo por T en todo el entrenamiento (se reutiliza para la corrección de priores)
    for T in T_GRID:
        Xb, yb = datos_T(Dv.Xtr[:, cols], Dv.ytr, T, Dv.semilla)
        m = nuevo_xgb(Dv.semilla).fit(Xb, yb)
        P = m.predict_proba(Dv.Xte[:, cols])
        clave = str(T or 0)
        res['curva_T'][clave] = indicadores(Dv.yte, P.argmax(1))
        if T:
            pi_b = np.bincount(yb, minlength=N_CLASES) / len(yb)
            res['priores'][clave] = indicadores(Dv.yte, corregir_priores(P, pi, pi_b))
        else:
            res['priores'][clave] = res['curva_T'][clave]
        print(f"   T = {T_txt(T):>12}: F1 {res['curva_T'][clave]['f1_macro']:.2f} % · TFP {res['curva_T'][clave]['tfp']:.3f} %"
              + (f"  | con priores: F1 {res['priores'][clave]['f1_macro']:.2f} % · TFP {res['priores'][clave]['tfp']:.3f} %" if T else ''))
    # 2) CV con restricción de TFP
    sub = submuestra(Dv.ytr, CV_SUBMUESTRA, Dv.semilla)
    cv = []
    for T in T_GRID:
        f1, tfp = cv_f1_tfp(Dv, cols, T, sub); cv.append({'T': T or 0, 'f1_cv': f1, 'tfp_cv': tfp})
    tfp_sin = next(c['tfp_cv'] for c in cv if c['T'] == 0)
    admisibles = [c for c in cv if c['tfp_cv'] <= tfp_sin]
    T_r = max(admisibles, key=lambda c: (c['f1_cv'], -c['T']))['T']
    res['cv_restringida'] = {'cv': cv, 'T': T_r, **res['curva_T'][str(T_r)]}
    # 3) Corrección de priores sobre el T que eligió el 07
    res['priores_T07'] = {'T': T07 or 0, **res['priores'][str(T07 or 0)]}
    res['segundos'] = round(time.time() - t_ini, 1)
    guardar_json(res, ruta)
    return res

if correr('A7'):
    EK = elegir_k(); A7 = {}
    for r in REPS:
        print(f'REP-{r:02d}'); A7[r] = a7_rep(r)
        print(f"   → CV restringida: T = {T_txt(A7[r]['cv_restringida']['T'])} · F1 {A7[r]['cv_restringida']['f1_macro']:.2f} % · "
              f"TFP {A7[r]['cv_restringida']['tfp']:.3f} %  | priores (T del 07 = {T_txt(A7[r]['T_07'])}): "
              f"F1 {A7[r]['priores_T07']['f1_macro']:.2f} % · TFP {A7[r]['priores_T07']['tfp']:.3f} %")

## A8. Resumen y pruebas pareadas

In [ ]:
if correr('A7'):
    ref = {r: leer_json(os.path.join(DIR_X, 'A2_arboles', f'REP-{r:02d}', 'resultado.json')) for r in REPS}
    n_tr, n_te = ref[REPS[0]]['n_train'], ref[REPS[0]]['n_test']
    IND = ('f1_macro', 'tfp', 'ataques_como_normal_pct', 'exactitud')
    series = {'MI-LiteXGB (07)': [ref[r]['modelos']['xgb'] for r in REPS],
              'MI-LiteRF (07)': [ref[r]['modelos']['rf'] for r in REPS],
              'XGB · CV con restricción de TFP': [A7[r]['cv_restringida'] for r in REPS],
              'XGB · corrección de priores': [A7[r]['priores_T07'] for r in REPS]}
    for T in T_GRID:
        series[f'XGB · T fijo = {T_txt(T)}'] = [A7[r]['curva_T'][str(T or 0)] for r in REPS]
        if T:
            series[f'XGB · T fijo = {T_txt(T)} + priores'] = [A7[r]['priores'][str(T or 0)] for r in REPS]
    resumen = {}
    for nombre, lst in series.items():
        resumen[nombre] = {k: ms([d[k] for d in lst]) for k in IND}
        resumen[nombre]['f1_por_clase'] = {c: float(np.mean([d['f1_por_clase'][c] for d in lst])) for c in CLASES}
        resumen[nombre]['matriz_sumada'] = np.sum([d['matriz_confusion'] for d in lst], axis=0).tolist()
        resumen[nombre]['n_rep'] = len(lst)
    resumen['XGB · CV con restricción de TFP']['T_elegidos'] = pd.Series(
        [T_txt(A7[r]['cv_restringida']['T'] or None) for r in REPS]).value_counts().to_dict()
    pruebas = []
    for ind in ('f1_macro', 'tfp'):
        bloque = []
        for var in ('XGB · CV con restricción de TFP', 'XGB · corrección de priores'):
            for base in ('MI-LiteXGB (07)', 'MI-LiteRF (07)'):
                d = [a[ind] - b[ind] for a, b in zip(series[var], series[base])]
                bloque.append({'indicador': ind, 'comparacion': f'{base} → {var}', **t_corregida(d, n_tr, n_te)})
        for p, ph in zip(bloque, holm([b['p_corr'] for b in bloque])):
            p['p_holm'] = float(ph)
        pruebas += bloque
    tabla = pd.DataFrame([{'variante': n, 'F1 macro': f"{v['f1_macro'][0]:.2f} ± {v['f1_macro'][1]:.2f}",
                           'TFP %': f"{v['tfp'][0]:.3f} ± {v['tfp'][1]:.3f}",
                           'ataques como Normal %': f"{v['ataques_como_normal_pct'][0]:.3f}"} for n, v in resumen.items()])
    mostrar(tabla)
    mostrar(pd.DataFrame(pruebas)[['indicador', 'comparacion', 'media_d', 'ic95_corr', 'p_holm']])
    S = {'generado': datetime.datetime.now().strftime('%d/%m/%Y %H:%M'), 'entorno': ENTORNO, 'tarea': TAREA, 'clases': CLASES,
         'reps': REPS, 'k': EK['k'], 'T_grid': [t or 0 for t in T_GRID], 'xgb': XGB_CFG,
         'reglas': {'cv_restringida': 'T admisible si TFP_CV ≤ TFP_CV(sin balanceo); se elige el de mayor F1_CV',
                    'priores': "p'(c|x) ∝ p(c|x)·π(c)/π_b(c) sobre el modelo con el T elegido en el 07 (Saerens et al., 2002)"},
         'resumen': resumen, 'pruebas': pruebas,
         'por_rep': [{'rep': r, 'T_07': A7[r]['T_07'], 'T_cv_restringida': A7[r]['cv_restringida']['T'],
                      'cv': A7[r]['cv_restringida']['cv'],
                      **{f'{n}|{k}': A7[r][n][k] for n in ('cv_restringida', 'priores_T07') for k in IND}} for r in REPS]}
    ruta = os.path.join(DIR_X, 'S14_XGB_TFP_resultados.json'); guardar_json(S, ruta)
    print('✔ Guardado:', ruta)